In [13]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Create the data/ folder if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# Set random seed for reproducibility
np.random.seed(42)

# Generate a simulated dataset (1000 records)
df = pd.DataFrame({
    'tenure': np.random.randint(1, 72, 1000),  # Customer tenure (1-72 months)
    'MonthlyCharges': np.random.uniform(20, 120, 1000),  # Monthly charges (20-120 EUR)
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], 1000),  # Contract type
    'Churn': np.random.choice([0, 1], 1000, p=[0.8, 0.2]),  # Churn label (20% churn probability)
    'latitude': np.random.uniform(48.1, 48.3, 1000),  # Latitude range for Vienna
    'longitude': np.random.uniform(16.2, 16.5, 1000)  # Longitude range for Vienna
})

# Basic data cleaning
df = df.dropna()
df['Churn'] = df['Churn'].astype(int)

# Display the first few rows
print(df.head())

# Save the data to the data/ folder
df.to_csv('data/simulated_churn_data.csv', index=False)

   tenure  MonthlyCharges        Contract  Churn   latitude  longitude
0      52      105.569647        One year      0  48.138145  16.217135
1      15      103.021986        One year      0  48.169412  16.241289
2      61       59.718353        Two year      1  48.209448  16.200279
3      21       86.808514  Month-to-month      0  48.298355  16.470967
4      24       40.498430        One year      0  48.232840  16.207385


In [14]:
# Import libraries for modeling
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Prepare features and target variable
features = ['tenure', 'MonthlyCharges', 'Contract']
X = pd.get_dummies(df[features], drop_first=True)  # Encode the categorical variable 'Contract'
y = df['Churn']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Random Forest model
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# Predict churn probability (for heatmap)
df['churn_probability'] = model.predict_proba(X)[:, 1]

# Evaluate the model
y_pred = model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

# Display data with churn probability
print(df[['tenure', 'MonthlyCharges', 'Churn', 'churn_probability']].head())

Test Accuracy: 0.735
   tenure  MonthlyCharges  Churn  churn_probability
0      52      105.569647      0               0.02
1      15      103.021986      0               0.12
2      61       59.718353      1               0.72
3      21       86.808514      0               0.08
4      24       40.498430      0               0.02


In [15]:
# Import folium library
import folium
from folium.plugins import HeatMap
import os
import numpy as np

# Ensure churn_probability is of float type
df['churn_probability'] = df['churn_probability'].astype(float)

# Adjust churn_probability distribution to increase the proportion of high-risk customers
# Using NumPy operations
df['adjusted_churn_probability'] = np.where(
    df['churn_probability'] > 0.5,
    np.sqrt(df['churn_probability']),  # Enhance the impact of high probability values
    df['churn_probability']
)

# Verify the range of adjusted probability values
print("Adjusted churn probability range:", df['adjusted_churn_probability'].min(), df['adjusted_churn_probability'].max())

# Create visualizations/ folder (already created, no need to repeat)
if not os.path.exists('visualizations'):
    os.makedirs('visualizations')

# Create a map centered on Vienna
m = folium.Map(location=[48.2082, 16.3738], zoom_start=12, tiles='OpenStreetMap')

# Prepare heatmap data: [latitude, longitude, adjusted_churn_probability]
heat_data = df[['latitude', 'longitude', 'adjusted_churn_probability']].values.tolist()

# Define custom color gradient using string keys to avoid folium errors
gradient = {
    '0.2': 'blue',    # Low risk: blue
    '0.4': 'green',   # Medium-low risk: green
    '0.6': 'yellow',  # Medium-high risk: yellow
    '1.0': 'red'      # High risk: red
}

# Add heatmap with adjusted parameters to highlight high-risk areas
HeatMap(
    heat_data,
    radius=12,  # Reduce point radius to focus on high-risk areas
    blur=15,    # Reduce blur for clearer boundaries
    max_zoom=13,
    gradient=gradient  # Use the corrected color gradient
).add_to(m)

# Save the heatmap as HTML
m.save('visualizations/churn_heatmap.html')

# Indicate successful save
print("Heatmap saved as visualizations/churn_heatmap.html")

Adjusted churn probability range: 0.0 0.9695359714832658
Heatmap saved as visualizations/churn_heatmap.html


In [12]:
### 业务洞察

- **高风险区域识别**：优化后的热力图显示维也纳北部（如 Leopoldstadt 和 Brigittenau）和南部（如 Favoriten）存在较高流失风险（黄色至红色区域）。建议在这些区域开展本地化促销活动，例如提供网络优化服务或折扣套餐。
- **客户特征分析**：模型预测表明，短期合约（Month-to-month）和高月费客户流失概率较高。可以通过推广长期合约优惠或调整高价套餐的定价策略降低流失率。
- **未来改进**：建议整合真实地理数据（如基站覆盖范围或客户反馈数据），进一步提高流失预测的准确性和区域分析的针对性。

SyntaxError: invalid character '：' (U+FF1A) (218344744.py, line 3)